In [1]:
#import necessory packages
import pandas as pd
import numpy as np

In [2]:
import sys
print(sys.executable)

d:\Dimple-AI\Guvi_Projects\Traffic_ViolationEDA\venv\Scripts\python.exe


In [4]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   ---------------------------------------- 0/2 [et-xmlfile]
   ---------------------------------------- 0/2 [et-xmlfile]
   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the k

In [5]:
# Load Excel file 
df = pd.read_excel(r"..\contents\Traffic_violation.xlsx")

In [6]:
#taking copy of the original 
df_copy = df


In [7]:
#rename Columns
df_copy = df_copy.rename(columns={'Date Of Stop' : 'Date_Of_Stop',
                                                          'Time Of Stop': 'Time_Of_Stop',
                                                          'Personal Injury' :'Personal_Injury',
                                                          'Property Damage' : 'Property_Damage',
                                                          'Commercial License' : 'Commercial_License',
                                                          'Commercial Vehicle' : 'Commercial_Vehicle',
                                                          'Work Zone': 'Work_Zone',
                                                          'Search Conducted' : 'Search_Conducted',
                                                          'Search Disposition' : 'Search_Disposition',
                                                          'Search Outcome' :'Search_Outcome',
                                                          'Search Reason' : 'Search_Reason',
                                                          'Search Reason For Stop' :'Search_Reason_For_Stop',
                                                          'Search Type' :'Search_Type',
                                                          'Search Arrest Reason' :'Search_Arrest_Reason',
                                                          'Violation Type': 'Violation_Type',
                                                          'Contributed To Accident' :'Contributed_To_Accident',
                                                          'Driver City' :'Driver_City',
                                                          'Driver State' : 'Driver_State',
                                                          'DL State' : 'DL_State',
                                                          'Arrest Type': 'Arrest_Type'})


In [8]:
agg_dict = {}
for col in df_copy.columns:
    if col == "SeqID":
        continue
    elif col == "Description" or col == "Charge":
        agg_dict[col] = lambda x: ", ".join(x.dropna().astype(str).unique())
    else:
        agg_dict[col] = "first"

summary_df = df_copy.groupby("SeqID", as_index=False).agg(agg_dict)

In [9]:
summary_df_temp = summary_df
summary_df["Charge"]

0                                           21-503(c)
1                                           21-801(b)
2         13-401(h), 13-411(f), 13-401(b1), 13-411(d)
3                                            22-201.1
4                           22-412.3(c2), 22-412.3(b)
                             ...                     
568052                                      13-401(h)
568053                                       21-801.1
568054                                             61
568055                                      13-409(b)
568056                                       21-801.1
Name: Charge, Length: 568057, dtype: object

In [ ]:
# Convert to datetime and handle invalid Date by replacing it to "Nan"
summary_df_temp['Date_Of_Stop'] = pd.to_datetime(summary_df_temp['Date_Of_Stop'],errors="coerce")
summary_df_temp["Time_Of_Stop"] = pd.to_datetime(summary_df_temp["Time_Of_Stop"],format="%H:%M:%S",
    errors="coerce") .dt.strftime("%H:%M:%S")

In [11]:
#Remove leading and trailing space from all columns except numeric columns
summary_df_temp = summary_df_temp.apply(
    lambda col: col.str.strip() if col.dtype == "object" else col
)

In [12]:
#Standardize @ or / separators. : replacing / with @
summary_df_temp["Location"] = (
    summary_df_temp["Location"]
    .str.replace("/", "@", regex=False)

)

In [13]:
# replace 0 with nan because 0 is invaild entry
summary_df_temp["Latitude"] = summary_df_temp["Latitude"].replace(0,np.nan, regex=False)
summary_df_temp["Longitude"] = summary_df_temp["Longitude"].replace(0,np.nan, regex=False)

In [14]:
# replace all the yes /no into True / false - convert columnn into boolen type 

summary_df_temp["Search_Conducted"] = summary_df_temp["Search_Conducted"].fillna("No")
boolType_cols= ["Accident","Belts", "Personal_Injury", "Property_Damage", "Fatal", 
                "Commercial_License", "HAZMAT", "Commercial_Vehicle", "Alcohol","Work_Zone","Search_Conducted"]
summary_df_temp[boolType_cols] = summary_df_temp[boolType_cols].apply(lambda col :(col.str.strip()
                                             .str.lower()
                                             .map({"yes" : True,
                                                   "no": False,
                                                   "" : False
                                                   }).astype("bool")))

In [15]:
# Fill empty value with "Not Applicable ""
summary_df_temp["Search_Disposition"] = summary_df_temp["Search_Disposition"].fillna("Not Applicable")
summary_df_temp["Search_Outcome"] = summary_df_temp["Search_Outcome"].fillna("Not Applicable")

In [16]:
# standardized Code
# Convert to uppercase
summary_df_temp["Search_Reason_For_Stop"] =  summary_df_temp["Search_Reason_For_Stop"].astype(str).str.strip().str.replace(r"\s+", "", regex=True).str.upper() 
summary_df_temp["VehicleType"] = summary_df_temp["VehicleType"].str.strip().str.replace(r"\s+","",regex=True).str.upper()
summary_df_temp["Search_Reason"] = summary_df_temp["Search_Reason"].fillna("No Reason Mentioned").astype(str).str.upper()
summary_df_temp["Search_Type"]   = summary_df_temp["Search_Type"].fillna("Not Applicable")                        
summary_df_temp["Search_Arrest_Reason"] = summary_df_temp["Search_Arrest_Reason"].fillna("Not Applicable").str.upper()                                      
summary_df_temp["State"] = summary_df_temp["State"].str.upper()
summary_df_temp["Model"] =  summary_df_temp["Model"].str.upper()

In [17]:
# Convert to numeric.
#Remove impossible years (<1960 or >2025).
#Check for nulls.

summary_df_temp["Year"] = pd.to_numeric(summary_df_temp["Year"], errors="coerce")
# Create a mask for years in the range
mask = summary_df_temp["Year"].between(1960, 2025, inclusive="both")
# Keep years in range as they are, set years outside range to NaN
summary_df_temp["Year"] = summary_df_temp["Year"].where(mask, np.nan)
# Use nullable integer type instead:
summary_df_temp["Year"] = summary_df_temp["Year"].astype("Int64")

# replace (0.0,0.0) geolocation  wit NaN
summary_df_temp['Geolocation'] = summary_df_temp['Geolocation'].replace('(0.0, 0.0)', np.nan)

In [ ]:
# Make Column - Correcting Typos
# Create a mapping of typos to correct values
typo_map = {
    'TOYT': 'TOYOTA',
    'NISS': 'NISSAN',
    'CHEV': 'CHEVROLET',
    'CHEVRO0LET' : "CHEVROLET",
    'CHEVY' : 'CHEVROLET',
    'HOND': 'HONDA',
    'HOND4S': 'HONDA',
    'TES1' : 'TESLA',
    'HYUN': 'HYUNDAI',
    'MERZ' : 'MERCEDES'
}

def typo_correction(make):
    if make is None or isinstance(make, float):
        return None
    make = make.upper().strip()  # Just use .strip(), not .map()
    return typo_map.get(make, make)  # Return mapped value or original

summary_df_temp['Make'] = summary_df_temp['Make'].apply(typo_correction)

# Get top 10 makes
top_10_makes = summary_df_temp['Make'].value_counts().head(10).index.tolist()

# Keep top 10, mark rest as "Others"
summary_df_temp['Make'] = summary_df_temp['Make'].apply(
    lambda x: x if x in top_10_makes else "Others"
)



In [22]:
print(len(summary_df_temp.columns))
print(summary_df_temp.columns.tolist())

values = list(summary_df_temp.itertuples(index=False, name=None))
print(len(values[0]))
print(values[0])

43
['SeqID', 'Date_Of_Stop', 'Time_Of_Stop', 'Agency', 'SubAgency', 'Description', 'Location', 'Latitude', 'Longitude', 'Accident', 'Belts', 'Personal_Injury', 'Property_Damage', 'Fatal', 'Commercial_License', 'HAZMAT', 'Commercial_Vehicle', 'Alcohol', 'Work_Zone', 'Search_Conducted', 'Search_Disposition', 'Search_Outcome', 'Search_Reason', 'Search_Reason_For_Stop', 'Search_Type', 'Search_Arrest_Reason', 'State', 'VehicleType', 'Year', 'Make', 'Model', 'Color', 'Violation_Type', 'Charge', 'Article', 'Contributed_To_Accident', 'Race', 'Gender', 'Driver_City', 'Driver_State', 'DL_State', 'Arrest_Type', 'Geolocation']
43
('00001e27-8bde-4328-8b33-2d2d9a9ce862', Timestamp('2015-04-11 00:00:00'), '10:54:00', 'MCP', '3rd District, Silver Spring', 'PEDESTRIAN CROSSING ROADWAY BETWEEN ADJACENT INTERSECTIONS HAVING TRAFFIC CONTROL SIGNAL', 'VEIRS MILL @ ENNALLS', 39.039675, -77.0544366666667, False, False, False, False, False, False, False, False, False, False, False, 'Not Applicable', 'Citatio

In [23]:
#save it to excel file 
summary_df_temp.head(500).to_excel("..\contents\Temp_Output.xlsx",index=False)

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\dimpl\AppData\Local\Temp\ipykernel_12468\3630468010.py:2: SyntaxWarning: invalid escape sequence '\c'
  summary_df_temp.head(500).to_excel("..\contents\Temp_Output.xlsx",index=False)


In [ ]:
import mysql.connector

# Function to Insert data into Table
def insertDataIntoSqlTable(df):
 # Step 1: Establish connection
    connection = mysql.connector.connect(
     host="localhost",
     user="root",
     password="sql@1234",
     database="traffic_violationsdb",
     autocommit=False
     )

 #create a cursor
    cursor = connection.cursor()
     # Execute Insert query
    query ="""INSERT INTO traffic_violation (SeqID,Date_Of_Stop,Time_Of_Stop,Agency,SubAgency,Description,
    Location,Latitude,Longitude,Accident,Belts,Personal_Injury,Property_Damage,Fatal,Commercial_License,
    HAZMAT,Commercial_Vehicle,Alcohol,Work_Zone,Search_Conducted,Search_Disposition,Search_Outcome,
    Search_Reason,Search_Reason_For_Stop,Search_Type,Search_Arrest_Reason,State,VehicleType,Year,
    Make,Model,Color,Violation_Type,Charge,Article,Contributed_To_Accident,Race,Gender,Driver_City,Driver_State,
    DL_State,Arrest_Type,Geolocation
    ) VALUES (
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
    );"""
    
    values = list(df.itertuples(index=False, name=None))

    batch_size = 1000

    for i in range(0, len(values), batch_size):
        batch = values[i:i + batch_size]
        cursor.executemany(query, batch)
        connection.commit()
        print(f"Inserted {i + len(batch)} rows")

   # Step 6: Close connection
    cursor.close()
    connection.close()

   

In [82]:
def clean_for_sql(df):
    df = df.copy()

    # Replace pandas NA / NaN with Python None
    df = df.astype(object).where(pd.notna(df), None)

    return df

clean_df = clean_for_sql(summary_df_temp)

insertDataIntoSqlTable(clean_df)

Inserted 1000 rows
Inserted 2000 rows
Inserted 3000 rows
Inserted 4000 rows
Inserted 5000 rows
Inserted 6000 rows
Inserted 7000 rows
Inserted 8000 rows
Inserted 9000 rows
Inserted 10000 rows
Inserted 11000 rows
Inserted 12000 rows
Inserted 13000 rows
Inserted 14000 rows
Inserted 15000 rows
Inserted 16000 rows
Inserted 17000 rows
Inserted 18000 rows
Inserted 19000 rows
Inserted 20000 rows
Inserted 21000 rows
Inserted 22000 rows
Inserted 23000 rows
Inserted 24000 rows
Inserted 25000 rows
Inserted 26000 rows
Inserted 27000 rows
Inserted 28000 rows
Inserted 29000 rows
Inserted 30000 rows
Inserted 31000 rows
Inserted 32000 rows
Inserted 33000 rows
Inserted 34000 rows
Inserted 35000 rows
Inserted 36000 rows
Inserted 37000 rows
Inserted 38000 rows
Inserted 39000 rows
Inserted 40000 rows
Inserted 41000 rows
Inserted 42000 rows
Inserted 43000 rows
Inserted 44000 rows
Inserted 45000 rows
Inserted 46000 rows
Inserted 47000 rows
Inserted 48000 rows
Inserted 49000 rows
Inserted 50000 rows
Inserted 